# Lab: Interrupted Time Series Counterfactual Validation

[View this lab on the QED Labs website](https://defenceeconomist.github.io/qedlabs/labs/interrupted-time-series-counterfactual-validation-lab.html)

## How To Use This Page

This is the third interrupted time-series lab. The guided core takes about 45–60 minutes; the predictor-support audit is an optional extension.

- Assume no credible untreated comparison series is available.
- Compare every candidate model on identical pre-rollout forecast origins.
- Do not use post-rollout outcomes to select the untreated model.
- Report the primary forecast and the closest competitor.


This lab uses a small seeded simulation because the untreated post-rollout outcome is known. The first two labs use real Seatbelts data; this lab trades realism for a counterfactual that can be checked directly.

## Training Goal

By the end of the core lab, you should be able to:

1. separate model selection data from intervention-period outcomes;
2. run identical rolling-origin checks at 3-, 6-, and 12-month horizons;
3. select a primary model using pre-intervention performance;
4. retain a near-performing comparator; and
5. report short-horizon observed-minus-expected gaps.

## Step 1: Generate A Minimal Known-Truth Case

The programme starts in January 2021. `untreated_outcome` is retained only so the lab can check the final forecasts; it must not enter model selection.

In [ ]:
required_packages <- c("ggplot2")
missing_packages <- required_packages[!vapply(
  required_packages,
  requireNamespace,
  logical(1),
  quietly = TRUE
)]
if (length(missing_packages) > 0) {
  install.packages(missing_packages, repos = "https://cloud.r-project.org")
}
invisible(lapply(required_packages, library, character.only = TRUE))

simulate_service <- function(seed = 48127L) {
  set.seed(seed)
  n_months <- 120L
  implementation_time <- 85L
  time <- seq_len(n_months)
  date <- seq(as.Date("2014-01-01"), by = "month", length.out = n_months)
  implemented <- as.integer(time >= implementation_time)
  post_time <- pmax(0L, time - implementation_time)
  season_sin <- sin(2 * pi * time / 12)
  season_cos <- cos(2 * pi * time / 12)

  context <- 100 + 0.10 * time + 4 * season_sin +
    as.numeric(arima.sim(list(ar = 0.45), n = n_months, sd = 1.2))

  pre_range <- range(context[time < implementation_time])
  supported_post <- time >= implementation_time & time <= 108L
  context[supported_post] <- pmin(
    pmax(context[supported_post], pre_range[[1]]),
    pre_range[[2]]
  )
  context[time >= 109L] <- context[time >= 109L] + 35

  untreated_noise <- as.numeric(arima.sim(list(ar = 0.35), n = n_months, sd = 2.2))
  untreated_outcome <- 120 + 0.18 * time + 7 * season_cos +
    0.9 * (context - 100) + untreated_noise
  transition <- as.integer(time %in% implementation_time:(implementation_time + 2L))
  programme_effect <- 18 * implemented + 0.35 * post_time - 6 * transition
  observed_outcome <- untreated_outcome + programme_effect

  data.frame(
    time,
    date,
    implemented,
    post_time,
    season_sin,
    season_cos,
    context,
    untreated_outcome,
    programme_effect,
    observed_outcome
  )
}

service <- simulate_service()
implementation_time <- min(service$time[service$implemented == 1L])
implementation_date <- min(service$date[service$implemented == 1L])
training_data <- subset(service, time < implementation_time)
primary_window <- subset(
  service,
  time >= implementation_time & time <= implementation_time + 11L
)

stopifnot(
  nrow(service) == 120L,
  implementation_date == as.Date("2021-01-01"),
  max(training_data$time) < implementation_time,
  nrow(primary_window) == 12L,
  all(is.finite(service$observed_outcome)),
  all(service$observed_outcome == service$untreated_outcome + service$programme_effect)
)

## Step 2: Define Three Candidate Untreated Models

The candidates encode different beliefs about what makes the series forecastable:

1. last year's observed month is sufficient;
2. a trend and annual cycle are sufficient; or
3. a prespecified contextual predictor adds useful untreated information.

In [ ]:
candidate_models <- c(
  "Seasonal naive",
  "Trend + season",
  "Context + trend"
)

forecast_candidate <- function(model_name, training, future) {
  if (model_name == "Seasonal naive") {
    lookup <- setNames(training$observed_outcome, training$time)
    prediction <- unname(lookup[as.character(future$time - 12L)])
  } else if (model_name == "Trend + season") {
    model <- lm(
      observed_outcome ~ time + season_sin + season_cos,
      data = training
    )
    prediction <- as.numeric(predict(model, newdata = future))
  } else if (model_name == "Context + trend") {
    model <- lm(
      observed_outcome ~ time + season_sin + season_cos + context,
      data = training
    )
    prediction <- as.numeric(predict(model, newdata = future))
  } else {
    stop("Unknown candidate model: ", model_name)
  }

  stopifnot(
    length(prediction) == nrow(future),
    all(is.finite(prediction))
  )
  prediction
}

No candidate includes `implemented`, `programme_effect`, post-rollout outcomes, or the hidden untreated outcome.

## Step 3: Validate On Identical Pre-Rollout Origins

Use the same three expanding-window origins for every model. At each origin, forecast the next 12 months and score the first 3, 6, and 12 months.

In [ ]:
validation_origins <- c(48L, 60L, 72L)
validation_horizons <- c(3L, 6L, 12L)

score_origin <- function(model_name, origin) {
  training <- subset(service, time <= origin)
  future <- subset(service, time > origin & time <= origin + 12L)
  predictions <- forecast_candidate(model_name, training, future)
  scale <- sd(training$observed_outcome)

  do.call(rbind, lapply(validation_horizons, function(horizon) {
    errors <- future$observed_outcome[seq_len(horizon)] - predictions[seq_len(horizon)]
    data.frame(
      model = model_name,
      origin = origin,
      horizon = horizon,
      normalized_RMSE = sqrt(mean(errors^2)) / scale
    )
  }))
}

validation_results <- do.call(rbind, lapply(candidate_models, function(model_name) {
  do.call(rbind, lapply(validation_origins, function(origin) {
    score_origin(model_name, origin)
  }))
}))

validation_summary <- aggregate(
  normalized_RMSE ~ model + horizon,
  data = validation_results,
  FUN = mean
)

validation_summary$normalized_RMSE <- round(validation_summary$normalized_RMSE, 3)
validation_summary[order(validation_summary$horizon, validation_summary$normalized_RMSE), ]

stopifnot(
  nrow(validation_results) ==
    length(candidate_models) * length(validation_origins) * length(validation_horizons),
  all(validation_results$origin + validation_results$horizon < implementation_time),
  all(is.finite(validation_results$normalized_RMSE))
)

Identical origins make the contest fair. The exercise tests forecast performance in untreated periods, not which model happens to produce the largest programme effect.

## Step 4: Select A Primary And Comparator Model

Average performance across the three reporting horizons. Keep the runner-up visible rather than treating a small rank difference as certainty.

In [ ]:
model_ranking <- aggregate(
  normalized_RMSE ~ model,
  data = validation_results,
  FUN = mean
)
model_ranking <- model_ranking[order(model_ranking$normalized_RMSE), ]
row.names(model_ranking) <- NULL

primary_model <- model_ranking$model[[1]]
comparator_model <- model_ranking$model[[2]]

model_ranking
data.frame(primary_model, comparator_model)

stopifnot(
  primary_model %in% candidate_models,
  comparator_model %in% candidate_models,
  primary_model != comparator_model
)

The ranking rule was fixed before looking at programme-period outcomes. If two models are close, that disagreement belongs in the result.

## Step 5: Forecast The First Twelve Programme Months

In [ ]:
selected_models <- c(primary_model, comparator_model)

impact_results <- do.call(rbind, lapply(selected_models, function(model_name) {
  prediction <- forecast_candidate(model_name, training_data, primary_window)
  data.frame(
    model = model_name,
    date = primary_window$date,
    observed = primary_window$observed_outcome,
    predicted_untreated = prediction,
    estimated_gap = primary_window$observed_outcome - prediction,
    true_untreated = primary_window$untreated_outcome,
    true_effect = primary_window$programme_effect
  )
}))

impact_summary <- aggregate(
  estimated_gap ~ model,
  data = impact_results,
  FUN = function(x) c(mean_gap = mean(x), cumulative_gap = sum(x))
)

impact_table <- data.frame(
  model = impact_summary$model,
  mean_monthly_gap = round(impact_summary$estimated_gap[, "mean_gap"], 1),
  cumulative_12_month_gap = round(impact_summary$estimated_gap[, "cumulative_gap"], 1)
)

impact_table

stopifnot(
  nrow(impact_results) == 2L * nrow(primary_window),
  all(is.finite(impact_results$predicted_untreated)),
  all(is.finite(impact_results$estimated_gap))
)

Only now reveal the known untreated values. Compare each model's estimated gap with `true_effect`; this is possible in the simulation but never in a real evaluation.

In [ ]:
truth_check <- aggregate(
  cbind(estimated_gap, true_effect) ~ model,
  data = impact_results,
  FUN = mean
)
truth_check$absolute_bias <- abs(truth_check$estimated_gap - truth_check$true_effect)
truth_check$estimated_gap <- round(truth_check$estimated_gap, 2)
truth_check$true_effect <- round(truth_check$true_effect, 2)
truth_check$absolute_bias <- round(truth_check$absolute_bias, 2)
truth_check

## Step 6: Plot Observed And Forecast Paths

In [ ]:
plot_history <- subset(service, time >= implementation_time - 36L)

ggplot() +
  geom_line(
    data = plot_history,
    aes(date, observed_outcome, colour = "Observed"),
    linewidth = 0.6
  ) +
  geom_line(
    data = impact_results,
    aes(date, predicted_untreated, colour = model),
    linewidth = 0.8,
    linetype = "dashed"
  ) +
  geom_vline(xintercept = implementation_date, linetype = "dotted") +
  scale_colour_manual(values = c(
    "Observed" = "#303030",
    "Seasonal naive" = "#8c6bb1",
    "Trend + season" = "#bf6b21",
    "Context + trend" = "#24527a"
  )) +
  labs(
    x = NULL,
    y = "Monthly outcome",
    colour = NULL,
    title = "Observed outcomes and preselected untreated forecasts"
  ) +
  theme_minimal(base_size = 12) +
  theme(legend.position = "bottom")

## Optional Extension: Stop When Predictor Support Fails

The contextual model relies on relationships learned within the pre-programme range. The simulation deliberately moves `context` far beyond that range after month 108.

In [ ]:
training_support <- range(training_data$context)

service$context_supported <-
  service$context >= training_support[[1]] &
  service$context <= training_support[[2]]

support_audit <- aggregate(
  context_supported ~ period,
  data = transform(
    subset(service, time >= implementation_time),
    period = ifelse(time <= 108L, "Primary supported period", "Later extension")
  ),
  FUN = mean
)

support_audit

first_unsupported_date <- min(service$date[
  service$time >= implementation_time & !service$context_supported
])
first_unsupported_date

stopifnot(
  all(primary_window$context >= training_support[[1]]),
  all(primary_window$context <= training_support[[2]]),
  first_unsupported_date == as.Date("2023-01-01")
)

If the contextual model is primary, stop its primary inference when support fails. Extra post-intervention observations do not justify extrapolating a predictor relationship into a context never seen during training.

## Suggested Close

Report:

1. the common validation origins and horizons;
2. the primary and comparator models;
3. their 12-month mean and cumulative gaps;
4. how closely each recovered the known effect; and
5. the date at which the contextual extension became unsupported.

The transferable rule is simple: choose the untreated model on untreated data, preserve meaningful model disagreement, and stop when the model leaves its empirical support.